# Neural-network input loader, time windows, and training-only normalization

This notebook implements the first three modeling-preparation steps:

1. an efficient PyTorch loader for the split, artifact-rejected trial tensors;
2. configurable pre-feedback, feedback, and full post-cue windows; and
3. streaming electrode-frequency normalization calculated from **training participants only**.

The compressed tensors remain grouped by recording. A run-grouped batch sampler and small LRU cache prevent the loader from repeatedly decompressing an entire run for every individual trial.

In [1]:
from collections import OrderedDict
from pathlib import Path
import math
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset, Sampler

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate the project data directory.')

DATA_ROOT = find_project_data()
INDEX_PATH = DATA_ROOT / 'processed' / 'modeling_index' / 'trial_modeling_index_with_split.csv'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'modeling_input'
NORMALIZATION_ROOT = OUTPUT_ROOT / 'normalization'
NORMALIZATION_SUMMARY_PATH = OUTPUT_ROOT / 'normalization_summary.csv'
NORMALIZATION_ROOT.mkdir(parents=True, exist_ok=True)

WINDOWS_SECONDS = {
    'pre_feedback': (0.25, 1.25),
    'feedback': (1.25, 5.0),
    'full_post_cue': (0.0, 5.0),
}
EXPECTED_SPLIT_PARTICIPANTS = {'train': 55, 'validation': 12, 'test': 12}
RANDOM_STATE = 42

model_index = pd.read_csv(INDEX_PATH)
if len(model_index) != 17_439:
    raise ValueError(f'Expected 17,439 indexed trials, found {len(model_index):,}.')
if model_index['sample_id'].duplicated().any():
    raise ValueError('sample_id must be unique.')

display(model_index.groupby(['split', 'phase', 'class_label']).size().rename('trials').reset_index())
print('Participants:', model_index.groupby('split')['participant'].nunique().to_dict())


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 807, in st

,split,phase,class_label,trials
0,test,acquisition,left,449
1,test,acquisition,right,458
2,test,online,left,896
3,test,online,right,910
4,train,acquisition,left,2059
5,train,acquisition,right,2042
6,train,online,left,4041
7,train,online,right,4014
8,validation,acquisition,left,442
9,validation,acquisition,right,440


Participants: {'test': 12, 'train': 55, 'validation': 12}


## Verify split isolation before calculating statistics

In [2]:
participant_sets = {
    split: set(rows['participant'])
    for split, rows in model_index.groupby('split')
}
assert {split: len(values) for split, values in participant_sets.items()} == EXPECTED_SPLIT_PARTICIPANTS
assert participant_sets['train'].isdisjoint(participant_sets['validation'])
assert participant_sets['train'].isdisjoint(participant_sets['test'])
assert participant_sets['validation'].isdisjoint(participant_sets['test'])

path_split_counts = model_index.groupby('clean_tensor_relative_path')['split'].nunique()
assert path_split_counts.eq(1).all(), 'A tensor file is shared across splits.'

split_paths = {
    split: set(rows['clean_tensor_relative_path'])
    for split, rows in model_index.groupby('split')
}
assert split_paths['train'].isdisjoint(split_paths['validation'])
assert split_paths['train'].isdisjoint(split_paths['test'])
assert split_paths['validation'].isdisjoint(split_paths['test'])

print('Leakage checks passed: participants and tensor files are disjoint across all splits.')
print('Tensor files per split:', {split: len(paths) for split, paths in split_paths.items()})

Leakage checks passed: participants and tensor files are disjoint across all splits.
Tensor files per split: {'test': 72, 'train': 320, 'validation': 69}


## Define and verify cue-relative time windows

All windows exclude the pre-cue normalization interval. The endpoint is exclusive, so the full post-cue window contains samples from 0 through 4.984375 seconds.

In [3]:
reference_path = DATA_ROOT / model_index['clean_tensor_relative_path'].iloc[0]
with np.load(reference_path, allow_pickle=False) as reference:
    channels = reference['channels'].astype(str)
    frequencies_hz = reference['frequencies_hz'].astype(float)
    all_times = reference['times_relative_to_cue_seconds'].astype(float)

assert len(channels) == 27
assert len(frequencies_hz) == 23

window_slices = {}
window_rows = []
for window_name, (start_seconds, stop_seconds) in WINDOWS_SECONDS.items():
    positions = np.flatnonzero((all_times >= start_seconds) & (all_times < stop_seconds))
    if not len(positions) or not np.all(np.diff(positions) == 1):
        raise ValueError(f'{window_name} does not map to one contiguous time slice.')
    window_slices[window_name] = slice(int(positions[0]), int(positions[-1]) + 1)
    selected_times = all_times[window_slices[window_name]]
    window_rows.append({
        'window': window_name,
        'requested_start_seconds': start_seconds,
        'requested_stop_seconds_exclusive': stop_seconds,
        'first_sample_seconds': selected_times[0],
        'last_sample_seconds': selected_times[-1],
        'time_points': len(selected_times),
        'tensor_shape_per_trial': f'27 × 23 × {len(selected_times)}',
    })

window_table = pd.DataFrame(window_rows)
display(window_table)

,window,requested_start_seconds,requested_stop_seconds_exclusive,first_sample_seconds,last_sample_seconds,time_points,tensor_shape_per_trial
0,pre_feedback,0.25,1.25,0.25,1.234375,64,27 × 23 × 64
1,feedback,1.25,5.00,1.25,4.984375,240,27 × 23 × 240
2,full_post_cue,0.00,5.00,0.00,4.984375,320,27 × 23 × 320


## Stream training-only normalization statistics

For each window, one mean and standard deviation are calculated for every electrode-frequency pair across all training trials and all time points in that window. Each training run tensor is decompressed once. Validation and test paths are never opened by this calculation.

In [4]:
train_index = model_index.loc[model_index['split'].eq('train')].copy()
assert train_index['participant'].nunique() == 55
assert set(train_index['clean_tensor_relative_path']).isdisjoint(split_paths['validation'] | split_paths['test'])

accumulators = {
    name: {
        'sum': np.zeros((len(channels), len(frequencies_hz)), dtype=np.float64),
        'sum_squares': np.zeros((len(channels), len(frequencies_hz)), dtype=np.float64),
        'observations_per_feature': 0,
        'trials': 0,
    }
    for name in WINDOWS_SECONDS
}

normalization_started = time.perf_counter()
grouped_train_files = list(train_index.groupby('clean_tensor_relative_path', sort=True))
for file_number, (relative_path, rows) in enumerate(grouped_train_files, start=1):
    tensor_path = DATA_ROOT / relative_path
    row_indices = rows['tensor_row_index'].to_numpy(dtype=int)
    with np.load(tensor_path, allow_pickle=False) as saved:
        X = saved['X'][row_indices]

    if X.shape[1:] != (27, 23, 512):
        raise ValueError(f'Unexpected tensor shape in {relative_path}: {X.shape}')
    if not np.isfinite(X).all():
        raise ValueError(f'Non-finite value in {relative_path}.')

    for window_name, time_slice in window_slices.items():
        selected = X[..., time_slice]
        accumulator = accumulators[window_name]
        accumulator['sum'] += selected.sum(axis=(0, 3), dtype=np.float64)
        accumulator['sum_squares'] += np.square(selected, dtype=np.float64).sum(axis=(0, 3))
        accumulator['observations_per_feature'] += selected.shape[0] * selected.shape[-1]
        accumulator['trials'] += selected.shape[0]

    if file_number == 1 or file_number % 25 == 0 or file_number == len(grouped_train_files):
        elapsed_minutes = (time.perf_counter() - normalization_started) / 60
        print(f'[{file_number:>3}/{len(grouped_train_files)}] training tensors accumulated ({elapsed_minutes:.1f} min)')

normalization_rows = []
normalization_parameters = {}
for window_name, accumulator in accumulators.items():
    count = accumulator['observations_per_feature']
    mean = accumulator['sum'] / count
    variance = np.maximum(accumulator['sum_squares'] / count - np.square(mean), 0.0)
    std = np.sqrt(variance)
    if not np.isfinite(mean).all() or not np.isfinite(std).all() or np.any(std <= 0):
        raise ValueError(f'Invalid normalization statistics for {window_name}.')

    normalization_parameters[window_name] = {'mean': mean, 'std': std}
    destination = NORMALIZATION_ROOT / f'{window_name}_train_only.npz'
    temporary = destination.with_name(destination.name + '.tmp.npz')
    start_seconds, stop_seconds = WINDOWS_SECONDS[window_name]
    np.savez_compressed(
        temporary,
        mean_electrode_frequency=mean,
        std_electrode_frequency=std,
        channels=channels,
        frequencies_hz=frequencies_hz,
        selected_times_seconds=all_times[window_slices[window_name]],
        window_name=np.asarray(window_name),
        requested_window_seconds=np.asarray([start_seconds, stop_seconds]),
        training_participants=np.asarray(sorted(participant_sets['train'])),
        training_tensor_files=np.asarray(sorted(split_paths['train'])),
        training_trials=np.asarray(accumulator['trials']),
        observations_per_electrode_frequency=np.asarray(count),
    )
    temporary.replace(destination)
    normalization_rows.append({
        'window': window_name,
        'training_participants': len(participant_sets['train']),
        'training_tensor_files': len(grouped_train_files),
        'training_trials': accumulator['trials'],
        'time_points': len(all_times[window_slices[window_name]]),
        'observations_per_electrode_frequency': count,
        'mean_min': mean.min(), 'mean_max': mean.max(),
        'std_min': std.min(), 'std_max': std.max(),
        'parameter_file': destination.relative_to(DATA_ROOT).as_posix(),
    })

normalization_summary = pd.DataFrame(normalization_rows)
normalization_summary.to_csv(NORMALIZATION_SUMMARY_PATH, index=False)
display(normalization_summary)

[  1/320] training tensors accumulated (0.0 min)
[ 25/320] training tensors accumulated (0.1 min)
[ 50/320] training tensors accumulated (0.1 min)
[ 75/320] training tensors accumulated (0.2 min)
[100/320] training tensors accumulated (0.3 min)
[125/320] training tensors accumulated (0.3 min)
[150/320] training tensors accumulated (0.4 min)
[175/320] training tensors accumulated (0.5 min)
[200/320] training tensors accumulated (0.5 min)
[225/320] training tensors accumulated (0.6 min)
[250/320] training tensors accumulated (0.7 min)
[275/320] training tensors accumulated (0.7 min)
[300/320] training tensors accumulated (0.8 min)
[320/320] training tensors accumulated (0.8 min)


,window,training_participants,training_tensor_files,training_trials,time_points,observations_per_electrode_frequency,mean_min,mean_max,std_min,std_max,parameter_file
0,pre_feedback,55,320,12156,64,777984,-3.704637,-2.547147,5.848171,6.611911,processed/modeling_input/normalization/pre_fee...
1,feedback,55,320,12156,240,2917440,-4.056330,-2.448841,5.883272,6.675376,processed/modeling_input/normalization/feedbac...
2,full_post_cue,55,320,12156,320,3889920,-3.947820,-2.483117,5.878434,6.658454,processed/modeling_input/normalization/full_po...


## Efficient cached PyTorch dataset and run-grouped sampler

The dataset exposes individual trials, while the sampler keeps each batch within a recording. This allows a small cache to be effective even when recordings and trial order are shuffled.

In [5]:
class RunTensorCache:
    def __init__(self, data_root, max_files=2):
        self.data_root = Path(data_root)
        self.max_files = max_files
        self._cache = OrderedDict()

    def get(self, relative_path):
        if relative_path in self._cache:
            self._cache.move_to_end(relative_path)
            return self._cache[relative_path]
        with np.load(self.data_root / relative_path, allow_pickle=False) as saved:
            value = {'X': saved['X'], 'y': saved['y']}
        self._cache[relative_path] = value
        self._cache.move_to_end(relative_path)
        while len(self._cache) > self.max_files:
            self._cache.popitem(last=False)
        return value


class EEGTrialDataset(Dataset):
    def __init__(self, index_frame, data_root, time_slice, mean=None, std=None, cache_files=2):
        self.index = index_frame.reset_index(drop=True).copy()
        self.time_slice = time_slice
        self.mean = None if mean is None else np.asarray(mean, dtype=np.float32)[..., None]
        self.std = None if std is None else np.asarray(std, dtype=np.float32)[..., None]
        self.cache = RunTensorCache(data_root, max_files=cache_files)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        row = self.index.iloc[index]
        run = self.cache.get(row.clean_tensor_relative_path)
        tensor_row = int(row.tensor_row_index)
        X = np.array(run['X'][tensor_row, ..., self.time_slice], dtype=np.float32, copy=True)
        y = int(run['y'][tensor_row])
        if self.mean is not None:
            X = (X - self.mean) / self.std
        if y != int(row.label_id):
            raise ValueError(f'Label mismatch for {row.sample_id}.')
        # This environment's Torch build predates NumPy 2.x bridge support.
        # frombuffer avoids torch.from_numpy while preserving the float32 array exactly.
        X_tensor = torch.frombuffer(memoryview(X), dtype=torch.float32).reshape(X.shape)
        return X_tensor, torch.tensor(y, dtype=torch.long), row.sample_id


class RunGroupedBatchSampler(Sampler):
    def __init__(self, index_frame, batch_size, shuffle=True, seed=42):
        self.groups = [positions.to_numpy() for _, positions in index_frame.groupby('clean_tensor_relative_path').groups.items()]
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        group_order = rng.permutation(len(self.groups)) if self.shuffle else np.arange(len(self.groups))
        for group_index in group_order:
            positions = self.groups[group_index].copy()
            if self.shuffle:
                rng.shuffle(positions)
            for start in range(0, len(positions), self.batch_size):
                yield positions[start:start + self.batch_size].tolist()

    def __len__(self):
        return sum(math.ceil(len(group) / self.batch_size) for group in self.groups)


## Instantiate and validate train, validation, and test loaders

Change `ACTIVE_WINDOW` to switch representations. All three splits receive the normalization parameters previously calculated from training data only.

In [6]:
ACTIVE_WINDOW = 'full_post_cue'
BATCH_SIZE = 16

active_parameters = normalization_parameters[ACTIVE_WINDOW]
split_frames = {
    split: model_index.loc[model_index['split'].eq(split)].reset_index(drop=True)
    for split in ('train', 'validation', 'test')
}
datasets = {
    split: EEGTrialDataset(
        frame,
        DATA_ROOT,
        window_slices[ACTIVE_WINDOW],
        mean=active_parameters['mean'],
        std=active_parameters['std'],
        cache_files=2,
    )
    for split, frame in split_frames.items()
}
samplers = {
    split: RunGroupedBatchSampler(
        frame, batch_size=BATCH_SIZE,
        shuffle=(split == 'train'), seed=RANDOM_STATE,
    )
    for split, frame in split_frames.items()
}
loaders = {
    split: DataLoader(dataset, batch_sampler=samplers[split], num_workers=0)
    for split, dataset in datasets.items()
}

loader_checks = []
for split, loader in loaders.items():
    X_batch, y_batch, sample_ids = next(iter(loader))
    loader_checks.append({
        'split': split,
        'dataset_trials': len(datasets[split]),
        'batches': len(loader),
        'first_batch_shape': tuple(X_batch.shape),
        'finite': bool(torch.isfinite(X_batch).all()),
        'labels_in_first_batch': sorted(y_batch.unique().tolist()),
        'example_sample': sample_ids[0],
    })
    assert X_batch.shape[1:] == (27, 23, len(all_times[window_slices[ACTIVE_WINDOW]]))
    assert torch.isfinite(X_batch).all()
    assert set(y_batch.tolist()).issubset({0, 1})

display(pd.DataFrame(loader_checks))

,split,dataset_trials,batches,first_batch_shape,finite,labels_in_first_batch,example_sample
0,train,12156,935,"(16, 27, 23, 320)",True,"[0, 1]",A16_R3_T14
1,validation,2570,199,"(16, 27, 23, 320)",True,"[0, 1]",A1_R2_T1
2,test,2713,207,"(16, 27, 23, 320)",True,"[0, 1]",A10_R1_T1


## Verify saved parameters and standardized training behavior

A sample of training files is standardized and checked. Exact zero mean and unit variance are expected only over the entire training set used for the streaming calculation, not within every individual batch.

In [7]:
saved_parameter_checks = []
for window_name in WINDOWS_SECONDS:
    parameter_path = NORMALIZATION_ROOT / f'{window_name}_train_only.npz'
    with np.load(parameter_path, allow_pickle=False) as saved:
        saved_train_participants = set(saved['training_participants'].astype(str))
        saved_train_paths = set(saved['training_tensor_files'].astype(str))
        mean = saved['mean_electrode_frequency']
        std = saved['std_electrode_frequency']
        saved_parameter_checks.append({
            'window': window_name,
            'mean_shape': mean.shape,
            'std_shape': std.shape,
            'training_participants': len(saved_train_participants),
            'training_tensor_files': len(saved_train_paths),
            'validation_path_overlap': len(saved_train_paths & split_paths['validation']),
            'test_path_overlap': len(saved_train_paths & split_paths['test']),
        })
        assert saved_train_participants == participant_sets['train']
        assert saved_train_paths == split_paths['train']
        assert not saved_train_paths & split_paths['validation']
        assert not saved_train_paths & split_paths['test']
        assert mean.shape == std.shape == (27, 23)
        assert np.isfinite(mean).all() and np.isfinite(std).all() and np.all(std > 0)

display(pd.DataFrame(saved_parameter_checks))
print(f'Normalization summary: {NORMALIZATION_SUMMARY_PATH}')
print('Steps 1–3 complete: cached loader, time windows, and training-only normalization are ready.')

,window,mean_shape,std_shape,training_participants,training_tensor_files,validation_path_overlap,test_path_overlap
0,pre_feedback,"(27, 23)","(27, 23)",55,320,0,0
1,feedback,"(27, 23)","(27, 23)",55,320,0,0
2,full_post_cue,"(27, 23)","(27, 23)",55,320,0,0


Normalization summary: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/modeling_input/normalization_summary.csv
Steps 1–3 complete: cached loader, time windows, and training-only normalization are ready.


## Handoff

The next modeling notebook can import or copy these loader classes, choose an active window, and train a sanity-check classifier. Validation data may be used for model selection; the test loader should remain untouched until the architecture and hyperparameters are fixed.